In [2]:

from box import Box
import torch
import esm_adapterH
import yaml
from collections import OrderedDict


def load_checkpoints(model,checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    #this model does not contain esm2
    new_ordered_dict = OrderedDict()
    for key, value in checkpoint['state_dict1'].items():
            key = key.replace("esm2.","")
            new_ordered_dict[key] = value
    model.load_state_dict(new_ordered_dict, strict=False)
    print("checkpoints were loaded from " + checkpoint_path)


def extract_emb_perseq(model, alphabet, sequence):
    batch_converter = alphabet.get_batch_converter()
    data = [
        ("protein1", sequence),
    ]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    if torch.cuda.is_available():
         batch_tokens=batch_tokens.cuda()
    with torch.no_grad():
        wt_representation = model(batch_tokens,repr_layers=[model.num_layers])["representations"][model.num_layers]
    wt_representation = wt_representation.squeeze(0) #only one sequence a time
    return wt_representation

def load_model(config_path, model_location):
    with open(config_path) as file:
        config_file = yaml.full_load(file)
        configs = Box(config_file)
    model, alphabet = esm_adapterH.pretrained.esm2_t33_650M_UR50D(configs.model.esm_encoder.adapter_h)
    load_checkpoints(model, model_location)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
        print("Transferred model to GPU")
    return model,alphabet

In [ ]:
model_location = './checkpoint/checkpoint_best_val_rmsf_cor.pth'
model_config = './config/config_vivit3.yaml'
input_data = [
    "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG",
    "KALTARQQEVFDLIRDHISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE"
]

model, alphabet = load_model(model_config, model_location)
result=[]
for seq in input_data:
    result.append(extract_emb_perseq(model, alphabet, seq))

In [ ]:
from model_idr import load_model_idr
config_path='./config/idr_config_30CAID2_trainfix_adp16_adp4.yaml'
model_location='./checkpoint/idr.pth'

model = load_model_idr(config_path, model_location)

input_data = [
    "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG",
    "KALTARQQEVFDLIRDHISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE"
]
result=model([input_data[0]])

c:\Users\yjm85\Anaconda3\envs\test\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yjm85\OneDrive\Document\work\DPLM_clean\self\utils\utils.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We r

In [7]:
from model_ddt import load_model_ddt
config_path='./config/ddt_config_adapterH16_adapterH4.yaml'
model_location='./checkpoint/ddt.pth'

model = load_model_ddt(config_path, model_location)

wild_seq = ["MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"]
mut_seq = ["MKTVRQERLKSIVRILERSKEPVSGAQLKEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"]

result=model(wild_seq, mut_seq)

c:\Users\yjm85\OneDrive\Document\work\DPLM_clean\self\utils\utils.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_checkpoint = torch.load(model_location, map_loca

In [8]:
result

tensor([-0.0075], device='cuda:0', grad_fn=<SqueezeBackward1>)

In [6]:
from utils.utils import *
from model_ddt import *
config_path='./config/ddt_config_adapterH16_adapterH4.yaml'
model_location='./checkpoint/ddt.pth'
with open(config_path) as file:
        config_file = yaml.full_load(file)
        configs = Box(config_file)

In [2]:
from model_ddt import *
model = prepare_models(configs)


In [3]:
load_checkpoints_infer(model, model_location)

c:\Users\yjm85\OneDrive\Document\work\DPLM_clean\self\utils\utils.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_checkpoint = torch.load(model_location, map_loca

Encoder(
  (esm2): ESM2(
    (embed_tokens): Embedding(33, 1280, padding_idx=1)
    (layers): ModuleList(
      (0-16): 17 x TransformerAdapterLayer(
        (self_attn): MultiheadAttention(
          (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (rot_emb): RotaryEmbedding()
        )
        (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1280, out_features=5120, bias=True)
        (fc2): Linear(in_features=5120, out_features=1280, bias=True)
        (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      )
      (17-28): 12 x TransformerAdapterLayer(
        (self_attn): MultiheadAttention(
          (k_proj): Linear(in_features=1280, out_featu

In [4]:
wt_seq="MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
mut_seq="MKTVRQERLKSIVRILERSKEPVSGAQLAEEKSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"

result=model([wt_seq],[mut_seq])

In [5]:
result

tensor([-0.0015], device='cuda:0', grad_fn=<SqueezeBackward1>)